# A Gentle Introduction to RAG Applications

This notebook creates a simple RAG (Retrieval-Augmented Generation) system to answer questions from a PDF document using an open-source model.

In [38]:
PDF_FILE = "influenza_rag/treating-influenza.pdf"

# We'll be using Llama 3.1 8B for this example.
MODEL = "llama3.1"

## Loading the PDF document

Let's start by loading the PDF document and breaking it down into separate pages.

<img src='images/documents.png' width="1000">

In [39]:
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader(PDF_FILE)
pages = loader.load()

print(f"Number of pages: {len(pages)}")
print(f"Length of a page: {len(pages[1].page_content)}")
print("Content of a page:", pages[1].page_content)

Number of pages: 2
Length of a page: 4460
Content of a page: CS HCVG-15-FLU-101        September 10, 2022 What are the possible side effects of antiviral drugs?
Side effects vary for each medication. For example, the most common 
side effects for oseltamivir are nausea and vomiting, zanamivir can cause wheezing and difficulty breathing (bronchospasm), and peramivir can  
cause diarrhea.  
Other less common side effects also have been reported. Your health care provider can give you more information about these drugs, or you can check the Food and Drug Administration (FDA) website for specific information about antiviral drugs, including the manufacturer’s package insert. 
When should antiviral drugs be taken for treatment?
Studies show that flu antiviral drugs work best for treatment when started within two days of getting sick. However, starting them later can still be helpful, especially if the sick person is in a group at high risk for serious complications (see list in sidebar) or 

## Splitting the pages in chunks

Pages are too long, so let's split pages into different chunks.

<img src='images/splitter.png' width="1000">


In [40]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(chunk_size=1500, chunk_overlap=100)

chunks = splitter.split_documents(pages)
print(f"Number of chunks: {len(chunks)}")
print(f"Length of a chunk: {len(chunks[1].page_content)}")
print("Content of a chunk:", chunks[1].page_content)


Number of chunks: 7
Length of a chunk: 1449
Content of a chunk: What should I do if I think I have flu?
Check with your doctor promptly if you are in a group at high risk for serious complications and you get flu symptoms. Symptoms of flu can include fever, cough, sore throat, runny or stuffy nose, body aches, headache, chills, and fatigue. If you get flu, antiviral drugs are a treatment option. Your doctor may prescribe antiviral drugs to treat your flu illness. 
Should I still get a flu vaccine?
Yes. Antiviral drugs are not a substitute for getting a flu vaccine. While flu vaccines can vary in how they work, flu vaccination is the first and best way to prevent influenza. You should receive flu vaccine every year. Antiviral drugs are a second line of defense to treat flu if you get sick.
What are the benefits of antiviral drugs?
Antiviral treatment works best when started within two days of getting symptoms. Antiviral drugs can lessen fever and other symptoms and shorten the time you 

## Storing the chunks in a vector store

We can now generate embeddings for every chunk and store them in a vector store.

<img src='images/vectorstore.png' width="1000">


In [41]:
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import OllamaEmbeddings

embeddings = OllamaEmbeddings(model=MODEL)
vectorstore = FAISS.from_documents(chunks, embeddings)

## Setting up a retriever

We can use a retriever to find chunks in the vector store that are similar to a supplied question.

<img src='images/retriever.png' width="1000">



In [42]:
retriever = vectorstore.as_retriever()
retriever.invoke("What is the document about?")

[Document(metadata={'source': 'influenza_rag/treating-influenza.pdf', 'page': 1}, page_content='CS HCVG-15-FLU-101        September 10, 2022 What are the possible side effects of antiviral drugs?\nSide effects vary for each medication. For example, the most common \nside effects for oseltamivir are nausea and vomiting, zanamivir can cause wheezing and difficulty breathing (bronchospasm), and peramivir can  \ncause diarrhea.  \nOther less common side effects also have been reported. Your health care provider can give you more information about these drugs, or you can check the Food and Drug Administration (FDA) website for specific information about antiviral drugs, including the manufacturer’s package insert. \nWhen should antiviral drugs be taken for treatment?\nStudies show that flu antiviral drugs work best for treatment when started within two days of getting sick. However, starting them later can still be helpful, especially if the sick person is in a group at high risk for seriou

## Configuring the model

We'll be using Ollama to load the local model in memory. After creating the model, we can invoke it with a question to get the response back.

<img src='images/model.png' width="1000">

In [43]:
from langchain_ollama import ChatOllama

model = ChatOllama(model=MODEL, temperature=0)
model.invoke("Who is the president of the United States?")

AIMessage(content='As of my last update in April 2023, Joe Biden is the President of the United States. However, please note that this information may have changed since then due to updates or changes in leadership.\n\nTo get the most current and accurate information, I recommend checking reputable news sources or official government websites for the latest updates on the presidency.', response_metadata={'model': 'llama3.1', 'created_at': '2026-05-15T15:49:21.942599902Z', 'message': {'role': 'assistant', 'content': ''}, 'done': True, 'done_reason': 'stop', 'total_duration': 663500980, 'load_duration': 155949115, 'prompt_eval_count': 19, 'prompt_eval_duration': 16916250, 'eval_count': 69, 'eval_duration': 442847729}, id='run-5e7a313e-ac95-4dcd-9a26-48c8564cfc22-0', usage_metadata={'input_tokens': 19, 'output_tokens': 69, 'total_tokens': 88})

## Parsing the model's response

The response from the model is an `AIMessage` instance containing the answer. We can extract the text answer by using the appropriate output parser. We can connect the model and the parser using a chain.

<img src='images/parser.png' width="1000">


In [44]:
from langchain_core.output_parsers import StrOutputParser

parser = StrOutputParser()

chain = model | parser 
print(chain.invoke("Who is the president of the United States?"))

As of my last update in April 2023, Joe Biden is the President of the United States. However, please note that this information may have changed since then due to updates or changes in leadership.

To get the most current and accurate information, I recommend checking reputable news sources or official government websites for the latest updates on the presidency.


## Setting up a prompt

In addition to the question we want to ask, we also want to provide the model with the context from the PDF file. We can use a prompt template to define and reuse the prompt we'll use with the model.


<img src='images/prompt.png' width="1000">

In [45]:
from langchain.prompts import PromptTemplate

template = """
You are an assistant that provides answers to questions based on
a given context. 

Answer the question based on the context. If you can't answer the
question, reply "I don't know".

Be as concise as possible and go straight to the point.

Context: {context}

Question: {question}
"""

prompt = PromptTemplate.from_template(template)
print(prompt.format(context="Here is some context", question="Here is a question"))


You are an assistant that provides answers to questions based on
a given context. 

Answer the question based on the context. If you can't answer the
question, reply "I don't know".

Be as concise as possible and go straight to the point.

Context: Here is some context

Question: Here is a question



## Adding the prompt to the chain

We can now chain the prompt with the model and the parser.

<img src='images/chain1.png' width="1000">

In [46]:
chain = prompt | model | parser

chain.invoke({
    "context": "Mary's sister is Lucy", 
    "question": "Who is Lucy's sister?"
})


'Mary.'

## Adding the retriever to the chain

Finally, we can connect the retriever to the chain to get the context from the vector store.

<img src='images/chain2.png' width="1000">

In [47]:
from operator import itemgetter

chain = (
    {
        "context": itemgetter("question") | retriever,
        "question": itemgetter("question"),
    }
    | prompt
    | model
    | parser
)

## Using the chain to answer questions

Finally, we can use the chain to ask questions that will be answered using the PDF document.

In [48]:
questions = [
    "What is the document about?",
    "Summarize the document in few sentences.",
    "What is the best way to treat influence?",
]

for question in questions:
    print(f"Question: {question}")
    print(f"Answer: {chain.invoke({'question': question})}")
    print("*************************\n")

Question: What is the document about?
Answer: The document is about treating influenza (flu), specifically providing information on antiviral drugs for people at high risk of serious complications.
*************************

Question: Summarize the document in few sentences.
Answer: The document provides information on treating influenza (flu) for people at high risk of serious complications. It recommends four FDA-approved antiviral drugs: oseltamivir, zanamivir, peramivir, and baloxavir. These medications can be used to treat flu symptoms, reduce the risk of hospitalization, and prevent serious complications when started within two days of getting sick.
*************************

Question: What is the best way to treat influence?
Answer: Antiviral drugs are a treatment option. They work best when started within two days of getting symptoms, can lessen fever and other symptoms, shorten the time you're sick by about one day, and prevent serious flu complications.
**********************